In [16]:
from pprint import pprint
import random
import pandas as pd
import numpy as np

import RC_scheduling as rcs

In [17]:
# Damage scenario selection
ds_sel = 'DS2'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Repair time discretization (hours)
repair_time_interval = 0.25  # 15 minutes

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():

    repair_time = float(row["fix time (hours)"])

    # Round up to the nearest interval
    repair_time = np.ceil(repair_time / repair_time_interval) * repair_time_interval

    time_reparation[str(row["Pipe ID"])] = {
        "t_r": repair_time
    }

print(f"{len(time_reparation)} repairs loaded.")
#time_reparation

106 repairs loaded.


In [ ]:
# Number of crews
n_crews = 1

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [19]:
# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# Export the new INP file with the controls
output_inp= f"BBM-EPS_{ds_sel}_restoration_{n_crews}RC(RS).inp"

In [20]:
# indexes = [24,14,87,10,2,78,75,4,39,104,9,77,3,89,23,7,81,29,43,101,6,11,30,73,8,5,0,42,44,98,55,41,1,15,61,74,91,21,32,64,26,94,80,16,88,18,84,72,58,17,66,69,38,86,33,96,25,56,40,70,46,92,62,50,31,35,27,71,28,93,12,82,67,60,83,63,99,22,97,13,103,100,59,47,95,37,53,105,20,85,90,79,76,52,36,34,45,54,65,57,102,48,68,19,49,51]
indexes = list(range(n_rep))
print(len(indexes))

106


In [21]:
# # List of indexes randomly ordered corresponding to the pipe IDs to be repaired
# indexes = list(range(len(pipe_ids)))
# random.shuffle(indexes)
# print(indexes[:20])

In [22]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_parquet(
    "TravelTime_"+ds_sel+".parquet"
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,1951,3414,4988,5251,3404,6005,1252,2408,3094,3293,...,538,5488,5567,5677,5778,5959,6041,854,869,892
1951,0.25,0.25,0.25,0.75,0.25,0.75,0.50,0.25,0.25,0.25,...,0.25,0.75,0.75,0.75,0.75,0.75,0.75,0.50,0.50,0.50
3414,0.25,0.25,0.25,0.50,0.25,0.75,0.25,0.25,0.25,0.25,...,0.25,0.50,0.75,0.50,0.75,0.75,0.75,0.25,0.25,0.25
4988,0.25,0.25,0.25,0.50,0.25,0.75,0.25,0.25,0.25,0.25,...,0.25,0.75,0.75,0.50,0.75,0.75,0.75,0.25,0.25,0.25
5251,0.75,0.50,0.50,0.25,0.75,0.25,0.50,0.50,0.75,0.50,...,0.50,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.50,0.50
3404,0.25,0.25,0.25,0.75,0.25,0.75,0.25,0.25,0.25,0.25,...,0.25,0.75,0.75,0.75,0.75,0.75,0.75,0.25,0.25,0.25


In [23]:
# #Create schedule where the repair crews have no breaks (24h shifts)
# schedule = rcs.create_schedule_NBT(
#     reparations=time_reparation, 
#     dmatrix=dmatrix_df, 
#     pipe_ids=pipe_ids, 
#     indexes=indexes, 
#     n_crews=n_crews
#     )

# schedule

In [24]:
# #Create schedule where the repair crews have realistic shifts (e.g. from 7:00 to 17:00)
schedule = rcs.create_schedule(
    reparations=time_reparation,
    dmatrix=dmatrix_df,
    pipe_ids=pipe_ids,
    indexes=indexes,
    n_crews=n_crews,
    initial_travel_time=0.25,
#    work_start=7.0,
#    work_end=17.0,
#    start_clock_hour=7.0
)

schedule[-20:]

,Order,Pipe,Crew,Travel,Repair,Start,Finish
86,87,4538,1,0.50,2.75,288.25,291.00
87,88,4584,2,0.50,2.75,291.25,294.00
88,89,4622,1,0.25,2.75,291.25,294.00
89,90,4721,3,0.25,2.75,291.25,294.00
90,91,4764,1,0.25,2.75,294.25,297.00
91,92,4882,2,0.25,2.75,294.25,297.00
92,93,4942,3,0.25,2.75,294.25,297.00
93,95,5042,2,0.25,2.75,297.25,314.25
94,96,5140,3,0.25,2.75,297.25,314.25
95,94,501,1,0.50,2.75,297.50,314.50


In [25]:
# Convert the hours into a readable day/hour/minute format
def hours_to_daytime(hours):
    day = int(hours // 24)
    hour = hours % 24

    h = int(hour)
    m = int(round((hour - h) * 60))

    if m == 60:
        h += 1
        m = 0

    return f"Day {day} - {h:02d}:{m:02d}"

In [26]:
# Converts the [start] and [finish] columns from "schedule" into a readable day/hour/minute format
schedule["Start readable"] = schedule["Start"].apply(hours_to_daytime)
schedule["Finish readable"] = schedule["Finish"].apply(hours_to_daytime)

schedule[
    [
        "Order",
        "Pipe",
        "Crew",
        "Travel",
        "Repair",
        "Start readable",
        "Finish readable"
    ]
]

,Order,Pipe,Crew,Travel,Repair,Start readable,Finish readable
0,1,1951,1,0.25,7.25,Day 0 - 00:45,Day 0 - 08:00
1,2,3414,2,0.25,7.25,Day 0 - 00:45,Day 0 - 08:00
2,3,4988,3,0.25,7.25,Day 0 - 00:45,Day 0 - 08:00
3,5,3404,2,0.25,5.75,Day 0 - 08:15,Day 1 - 04:15
4,4,5251,1,0.75,7.25,Day 0 - 08:45,Day 1 - 06:15
...,...,...,...,...,...,...,...
101,102,5959,1,0.25,2.75,Day 13 - 06:15,Day 13 - 09:00
102,103,6041,3,0.25,2.75,Day 13 - 08:30,Day 14 - 01:30
103,104,854,1,0.75,2.75,Day 13 - 09:45,Day 14 - 02:45
104,105,869,2,0.75,2.75,Day 13 - 09:45,Day 14 - 02:45


In [27]:
# print(schedule[:20])

In [28]:
new_controls = rcs.generate_controls(
    schedule=schedule,
    comment_lines=True
)

print("\n".join(new_controls[:30]))

; Pipe 1951
LINK 1951_A CLOSED AT TIME 0.75
LINK 1951_B CLOSED AT TIME 0.75
LINK 1951 OPEN AT TIME 8.00
; Pipe 3414
LINK 3414_A CLOSED AT TIME 0.75
LINK 3414_B CLOSED AT TIME 0.75
LINK 3414 OPEN AT TIME 8.00
; Pipe 4988
LINK 4988_A CLOSED AT TIME 0.75
LINK 4988_B CLOSED AT TIME 0.75
LINK 4988 OPEN AT TIME 8.00
; Pipe 3404
LINK 3404_A CLOSED AT TIME 8.25
LINK 3404_B CLOSED AT TIME 8.25
LINK 3404 OPEN AT TIME 28.25
; Pipe 5251
LINK 5251_A CLOSED AT TIME 8.75
LINK 5251_B CLOSED AT TIME 8.75
LINK 5251 OPEN AT TIME 30.25
; Pipe 6005
LINK 6005_A CLOSED AT TIME 8.75
LINK 6005_B CLOSED AT TIME 8.75
LINK 6005 OPEN AT TIME 28.75
; Pipe 1252
LINK 1252_A CLOSED AT TIME 28.50
LINK 1252_B CLOSED AT TIME 28.50
LINK 1252 OPEN AT TIME 48.50
; Pipe 2408
LINK 2408_A CLOSED AT TIME 29.50


In [29]:
# Apply function to generate INP with controls
controls_inp = rcs.write_inp_controls(
    input_inp=input_inp,
    output_inp=output_inp,
    schedule=schedule,
    new_controls=new_controls
)

print(f"Created: {controls_inp}")

Created: BBM-EPS_DS2_restoration_3RC(RS).inp


In [31]:
schedule[-20:]

,Order,Pipe,Crew,Travel,Repair,Start,Finish,Start readable,Finish readable
86,87,4538,1,0.50,2.75,288.25,291.00,Day 12 - 00:15,Day 12 - 03:00
87,88,4584,2,0.50,2.75,291.25,294.00,Day 12 - 03:15,Day 12 - 06:00
88,89,4622,1,0.25,2.75,291.25,294.00,Day 12 - 03:15,Day 12 - 06:00
89,90,4721,3,0.25,2.75,291.25,294.00,Day 12 - 03:15,Day 12 - 06:00
90,91,4764,1,0.25,2.75,294.25,297.00,Day 12 - 06:15,Day 12 - 09:00
91,92,4882,2,0.25,2.75,294.25,297.00,Day 12 - 06:15,Day 12 - 09:00
92,93,4942,3,0.25,2.75,294.25,297.00,Day 12 - 06:15,Day 12 - 09:00
93,95,5042,2,0.25,2.75,297.25,314.25,Day 12 - 09:15,Day 13 - 02:15
94,96,5140,3,0.25,2.75,297.25,314.25,Day 12 - 09:15,Day 13 - 02:15
95,94,501,1,0.50,2.75,297.50,314.50,Day 12 - 09:30,Day 13 - 02:30
